# Collecting norms, standard deviations, filtering the dataset and plotting distributions

This notebook intends to gather statistics for the full as well as a filtered dataset (norm per qualification, gender etc.). I do the exercise both with and without found anomalies. Then I create helper functions that calculate statistics for a random subset of 100 samples.

In [31]:
# I get faulty dependencies when using my local python 3.12. I use 3.10 instead and install the following dependencies to make dataprep work
# %pip install -U dataprep
# %pip install --upgrade pip setuptools
# %pip install --upgrade wheel
# %pip cache purge
# %pip install python-dotenv --use-pep517


In [32]:
from helpers.dep2pyodbc import dep2connection
import pandas as pd
import numpy as np
from dataprep.eda import create_report
from datetime import datetime, timedelta

In [33]:
con = dep2connection("CRH_DWH")

query = """
        SELECT FCAQuestionKey, fq.FCATestKey, ItemID, TimeSpent AS TimeSpentQuestion, dts.tekst AS TestStartTime,
		      dtf.tekst AS TestFinishTime, ft.CandidateKey, Gender, ChosenGender, ChosenLanguage, Qualification, AnswerSequence1, AnswerSequence2, AnswerSequence3
        FROM FactQuestionFCA fq
                JOIN FactTestFCA ft ON fq.FCATestKey=ft.FCATestKey
                JOIN DimCandidate dc ON ft.CandidateKey=dc.CandidateKey
	              JOIN DimTime dts ON ft.TestStartTimeKey=dts.TimeKey
		            JOIN DimTime dtf ON ft.TestFinishTimeKey=dtf.TimeKey
        """
df_fca = pd.read_sql(query, con)

df_fca.head()

pyodbc using windows


C:\Users\Mikel\AppData\Local\Temp\ipykernel_18724\505872341.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fca = pd.read_sql(query, con)


,FCAQuestionKey,FCATestKey,ItemID,TimeSpentQuestion,TestStartTime,TestFinishTime,CandidateKey,Gender,ChosenGender,ChosenLanguage,Qualification,AnswerSequence1,AnswerSequence2,AnswerSequence3
0,1,1,222,11.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,65CAFEA9-CF49-43ED-99AA-83AFE0539978,Qualification_Unknown,0,0,0
1,2,1,223,3.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,65CAFEA9-CF49-43ED-99AA-83AFE0539978,Qualification_Unknown,3,4,2
2,3,1,226,13.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,65CAFEA9-CF49-43ED-99AA-83AFE0539978,Qualification_Unknown,4,3,2
3,4,1,229,100.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,65CAFEA9-CF49-43ED-99AA-83AFE0539978,Qualification_Unknown,2,0,4
4,5,1,238,2.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,65CAFEA9-CF49-43ED-99AA-83AFE0539978,Qualification_Unknown,0,0,0


In [34]:
# report = create_report(df_fca)

In [35]:
# report

TODO:

- Change languages to a readable format using the csv given by Hudson
- Either drop rows/colums with missing values, or fill them
- Add a TimeSpentTest and TotalTimeAllQuestions column and check for discrepancies
- Plot CandAnswer distributions
- Check the odds of the least popular answer combinations (4 and 5 are most common, which set of answers are only given in less than 5% of the time)
- Check for correlations between times spent and qualification/gender
 

In [ ]:
languages = pd.read_csv('../files/csv/Language_codes.csv',delimiter=';') # TODO
languages.columns = ["ChosenLanguage","HudsonID"]
languages.head()

,ChosenLanguage,HudsonID
0,65CAFEA9-CF49-43ED-99AA-83AFE0539978,en-INT
1,FADAABC8-26DD-458B-B07E-BF96B4B3FCED,nl-BE
2,BE3C46AE-0192-4A9B-ACEE-37E51836F77C,fr-BE
3,BE39F135-6EF5-47E0-92D9-81B13BF9BEAA,en-GB
4,11BBB1BC-0CCD-4C9B-B0BE-5EFF9E634F75,en-US


In [37]:
df_fca = df_fca.merge(languages, how='left', on="ChosenLanguage")
df_fca['ChosenLanguage'] = df_fca['HudsonID']
df_fca.drop(columns=['HudsonID'],inplace=True)
df_fca.head()

,FCAQuestionKey,FCATestKey,ItemID,TimeSpentQuestion,TestStartTime,TestFinishTime,CandidateKey,Gender,ChosenGender,ChosenLanguage,Qualification,AnswerSequence1,AnswerSequence2,AnswerSequence3
0,1,1,222,11.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,en-INT,Qualification_Unknown,0,0,0
1,2,1,223,3.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,en-INT,Qualification_Unknown,3,4,2
2,3,1,226,13.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,en-INT,Qualification_Unknown,4,3,2
3,4,1,229,100.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,en-INT,Qualification_Unknown,2,0,4
4,5,1,238,2.0,9:31:54,9:32:12,544,Gender_Female,Gender_Female,en-INT,Qualification_Unknown,0,0,0


In [38]:
# Rename the qualification
df_fca['Qualification'] = df_fca['Qualification'].str.split(pat='_').str[1]
df_fca[['Qualification']].value_counts()

Qualification
Unknown          2347602
Master            265412
Bachelor          257799
Secondary         102545
PostGraduate       25482
Professional       18419
MBA                 9753
Vocational          6992
PHD                 1679
dtype: int64

In [39]:
df_fca['TestStartTime'] = pd.to_datetime(df_fca['TestStartTime'], format='%H:%M:%S').dt.time
df_fca['TestFinishTime'] = pd.to_datetime(df_fca['TestFinishTime'], format='%H:%M:%S').dt.time
df_fca

,FCAQuestionKey,FCATestKey,ItemID,TimeSpentQuestion,TestStartTime,TestFinishTime,CandidateKey,Gender,ChosenGender,ChosenLanguage,Qualification,AnswerSequence1,AnswerSequence2,AnswerSequence3
0,1,1,222,11.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0
1,2,1,223,3.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,3,4,2
2,3,1,226,13.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,4,3,2
3,4,1,229,100.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,2,0,4
4,5,1,238,2.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035700,2867540,150941,415,47.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,5,4,2
3035701,2867541,150941,416,81.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,4,3,5
3035702,2832571,148787,419,55.0,12:29:59,12:57:26,44615571,Gender_Female,Gender_Female,fr-LU,Unknown,4,3,5
3035703,2762630,145223,235,55.0,04:43:19,05:14:24,43603271,Gender_Male,Gender_Male,en-AU,Unknown,5,1,3


In [40]:
df_fca["TimeSpentTest"] = df_fca.apply(lambda row: 
    (datetime.combine(datetime.today(), row['TestFinishTime']) + timedelta(days=1) if row['TestFinishTime'] < row['TestStartTime'] else datetime.combine(datetime.today(), row['TestFinishTime'])) 
    - datetime.combine(datetime.today(), row['TestStartTime']), axis=1)

In [41]:
df_fca['TimeSpentTest'].describe()

count                      3035705
mean     0 days 00:40:11.966695051
std      0 days 00:47:26.481287321
min                0 days 00:00:00
25%                0 days 00:27:13
50%                0 days 00:37:31
75%                0 days 00:50:27
max                0 days 23:59:34
Name: TimeSpentTest, dtype: object

In [42]:
df_fca

,FCAQuestionKey,FCATestKey,ItemID,TimeSpentQuestion,TestStartTime,TestFinishTime,CandidateKey,Gender,ChosenGender,ChosenLanguage,Qualification,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpentTest
0,1,1,222,11.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0,0 days 00:00:18
1,2,1,223,3.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,3,4,2,0 days 00:00:18
2,3,1,226,13.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,4,3,2,0 days 00:00:18
3,4,1,229,100.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,2,0,4,0 days 00:00:18
4,5,1,238,2.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0,0 days 00:00:18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035700,2867540,150941,415,47.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,5,4,2,0 days 00:23:42
3035701,2867541,150941,416,81.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,4,3,5,0 days 00:23:42
3035702,2832571,148787,419,55.0,12:29:59,12:57:26,44615571,Gender_Female,Gender_Female,fr-LU,Unknown,4,3,5,0 days 00:27:27
3035703,2762630,145223,235,55.0,04:43:19,05:14:24,43603271,Gender_Male,Gender_Male,en-AU,Unknown,5,1,3,0 days 00:31:05


Set the chosen gender to Gender where it is None (using a temporary df for troubleshooting)

In [43]:
temp = df_fca

In [44]:
temp['ChosenGender'] = temp.ChosenGender.fillna(temp.Gender)

In [45]:
df_fca = temp

In [46]:
df_fca.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3035705 entries, 0 to 3035704
Data columns (total 15 columns):
 #   Column             Dtype          
---  ------             -----          
 0   FCAQuestionKey     int64          
 1   FCATestKey         int64          
 2   ItemID             int64          
 3   TimeSpentQuestion  float64        
 4   TestStartTime      object         
 5   TestFinishTime     object         
 6   CandidateKey       int64          
 7   Gender             object         
 8   ChosenGender       object         
 9   ChosenLanguage     object         
 10  Qualification      object         
 11  AnswerSequence1    int64          
 12  AnswerSequence2    int64          
 13  AnswerSequence3    int64          
 14  TimeSpentTest      timedelta64[ns]
dtypes: float64(1), int64(7), object(6), timedelta64[ns](1)
memory usage: 370.6+ MB


In [47]:
temp = df_fca
temp.TimeSpentTest = temp.TimeSpentTest.dt.total_seconds().astype(int)
temp.TimeSpentTest

0            18
1            18
2            18
3            18
4            18
           ... 
3035700    1422
3035701    1422
3035702    1647
3035703    1865
3035704    1865
Name: TimeSpentTest, Length: 3035705, dtype: int32

In [48]:
df_fca

,FCAQuestionKey,FCATestKey,ItemID,TimeSpentQuestion,TestStartTime,TestFinishTime,CandidateKey,Gender,ChosenGender,ChosenLanguage,Qualification,AnswerSequence1,AnswerSequence2,AnswerSequence3,TimeSpentTest
0,1,1,222,11.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0,18
1,2,1,223,3.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,3,4,2,18
2,3,1,226,13.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,4,3,2,18
3,4,1,229,100.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,2,0,4,18
4,5,1,238,2.0,09:31:54,09:32:12,544,Gender_Female,Gender_Female,en-INT,Unknown,0,0,0,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3035700,2867540,150941,415,47.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,5,4,2,1422
3035701,2867541,150941,416,81.0,12:36:06,12:59:48,45807761,Gender_Male,Gender_Male,de-LU,Unknown,4,3,5,1422
3035702,2832571,148787,419,55.0,12:29:59,12:57:26,44615571,Gender_Female,Gender_Female,fr-LU,Unknown,4,3,5,1647
3035703,2762630,145223,235,55.0,04:43:19,05:14:24,43603271,Gender_Male,Gender_Male,en-AU,Unknown,5,1,3,1865
